# Portfolio risk monitor and multi-asset operations layer — results

Reads `results/run.json` and the DuckDB store written by `python -m xrisk run`. Run `scripts/run_all.sh` first (or `--quick` for four days).

In [ ]:
import json, os, duckdb, pandas as pd, matplotlib.pyplot as plt
ROOT = os.path.abspath('..')
run = json.load(open(os.path.join(ROOT, 'results', 'run.json')))
con = duckdb.connect(os.path.join(ROOT, 'data', 'derived', 'xrisk.duckdb'), read_only=True)
days = pd.DataFrame(run['days']); days['date'] = pd.to_datetime(days['date'])
print(run['from'], run['to'], len(days), 'days; xops from', run['config']['xops_source'])

## The monitor under fault injection

In [ ]:
m = run['monitor']
print(f"{m['detected']}/{m['injected']} faults caught, median time to detect {m['median_ttd_s']:.0f} s, p90 {m['p90_ttd_s']:.0f} s; {m['false_alerts_per_clean_day']:.1f} alerts a day on {m['clean_days']} clean days")
pd.DataFrame(m['by_type']).T.sort_values('injected', ascending=False)

In [ ]:
con.execute("select rule, severity, count(*) n from alerts group by 1,2 order by 3 desc").df()

## The book, exposures and limits

In [ ]:
pos = con.execute("select p.strategy, i.asset_class, count(*) n, sum(abs(p.notional_usd))/1e6 gross_m, sum(p.notional_usd)/1e6 net_m from positions p join instruments i using (instrument_id) where p.phase='EOD' and p.date=(select max(date) from positions) group by 1,2 order by 1,2").df()
pos

In [ ]:
ax = days.set_index('date')[['gross', 'net', 'beta_dollars']].div(1e6).plot(figsize=(10, 3.5), title='exposure at the close ($m)'); ax.axhline(0, color='grey', lw=0.5)
days.set_index('date')[['var_param', 'var_hist']].div(1e6).plot(figsize=(10, 3), title='one-day 99% VaR ($m)')

In [ ]:
con.execute("select limit_name, status, count(*) n from limits group by 1,2 order by 1,2").df().pivot(index='limit_name', columns='status', values='n').fillna(0).astype(int)

## P&L attribution (exact by construction)

In [ ]:
p = con.execute('select date, strategy, asset_class, component, value from pnl').df(); p['date'] = pd.to_datetime(p['date'])
print('largest identity gap over the run:', days['identity_gap'].abs().max(), 'USD')
(p.assign(k=p.asset_class + ':' + p.component).groupby('k')['value'].sum() / 1e6).round(2).sort_values()

In [ ]:
(p.pivot_table(index='date', columns='strategy', values='value', aggfunc='sum').fillna(0).cumsum() / 1e6).plot(figsize=(10, 3.5), title='cumulative P&L by strategy ($m)')

## Margin, rolls, FX, credit

In [ ]:
print(json.dumps({k: v for k, v in run['margin'].items() if k != 'scan_ranges'}, indent=1))
pd.DataFrame(run['margin']['scan_ranges']).T

In [ ]:
con.execute('select * from rolls order by date').df()

In [ ]:
print(json.dumps(run['fx'], indent=1))
con.execute('select value_date, ccy, count(*) instructions, sum(abs(amount)) gross, abs(sum(amount)) net, sum(case when confirmed then 1 else 0 end) confirmed from fx_settlements group by 1,2 order by 1,2').df()

In [ ]:
rq = con.execute('select * from rfq_trades').df()
print(json.dumps(run['credit'], indent=1))
rq[['markup_pts', 'markup_bp_yield', 'cover', 'trace_dev_pts', 'report_delay_s']].describe().round(3)

## Checks

In [ ]:
con.execute('select phase, check_name, status, count(*) n from checks group by 1,2,3 order by 1,2,3').df().pivot_table(index=['phase', 'check_name'], columns='status', values='n', aggfunc='sum').fillna(0).astype(int)